# 1 amending src files

In [22]:
IN_DIR = 'res_dfs7'
OUT_DIR = 'res_dfs7'

In [23]:
import pandas as pd
from numpy import select
import json
import os


def ld(p):
    return pd.read_csv(p, sep=',')

In [24]:
def rm_col(ncols, c):
    if c in ncols:
        ncols.remove(c)
    return ncols

## cols

In [25]:
IOU_COLS_GDA = ["3_test_GDA_iou", "3_test/GDA/iou", "1_test/GDA/iou",  
                "2_test/GDA/iou", "1_test_GDA_iou", "2_test_GDA_iou"]
IOU_COLS_SYNT = ["3_test_SYNT_iou", "1_test/SYNT/iou", "2_test/SYNT/iou", 
            "3_test/SYNT/iou", "1_test_SYNT_iou", "2_test_SYNT_iou"]
IOU_COLS_DK = ["3_test_DK_iou", "1_test/DK/iou", "2_test/DK/iou", 
            "3_test/DK/iou", "1_test_DK_iou", "2_test_DK_iou"]

IOU_COLS = IOU_COLS_GDA + IOU_COLS_SYNT + IOU_COLS_DK
# SPLIT_COLUMNS = ['val', 'Sweep']
SCOLS = ["batch_size", "epochs", "warmup_epochs", "wd", "ema", "sub", 
        "ID", "loss", "lrdec", "lrenc", "mod_ph1", "mod_ph2", "Sweep",
        "val", "1_epoch", "2_epoch", "3_epoch", 'Runtime'] + IOU_COLS

def ld_scols(p):
    df = pd.read_csv(p, sep=',')
    cols = df.columns
    cols = list(set(cols) & set(SCOLS))
    ncols = list(set(cols) & set(SCOLS))
    df = df[df['State'] == 'finished']
    # if 'ID' in cols:
    #     df.set_index('ID', inplace=True, drop=False)
        # df.reindex('ID')
    if 'ph3' in p:
        ncols = rm_col(ncols, "2_test/DK/iou")
        ncols = rm_col(ncols, "2_test/SYNT/iou")
        ncols = rm_col(ncols, "2_test/GDA/iou")
        ncols = rm_col(ncols, "2_test_DK_iou")
        ncols = rm_col(ncols, "2_test_GDA_iou")
        ncols = rm_col(ncols, "2_test_SYNT_iou")
    if 'S' in p:
        ncols = rm_col(ncols, "1_test_DK_iou")
        ncols = rm_col(ncols, "1_test_GDA_iou")
        ncols = rm_col(ncols, "1_test_SYNT_iou")
    for col in cols:
        if '/' in col:
            # print('sign found in', col)
            df[col.replace('/', '_')] = df[col]
            df = df.drop(columns=[col])
            if col in ncols:
                ncols.remove(col)
            ncols.append(col.replace('/', '_'))
    # print(ncols)
    return df[ncols].fillna({'ema': False})

In [26]:
def drop_empty_cols(df: pd.DataFrame):
    return df.dropna(axis='columns')

## tr_val

In [27]:
def map_dict_val(d: dict, val, deff):
    for k, v in d.items():
        # print(k, v)
        if k in val:
            # print(len(val.split('_')))
            if ('_' in val and len(val.split('_')) > 3) or 'ph1_gda' in val:
                return 'gda'
            return v
    if 'ph1_gda' in val:
        return 'gda'
    return deff

In [28]:
fn_tr = {
    'ph2': 'subm',
    'ph3': 'dk',
    'dk': 'dk',
    'subs': 'sub',
    'S': 'sub',
}

def add_tr(df: pd.DataFrame, fn):
    df['train'] = map_dict_val(fn_tr, fn, 's')
    return df

In [29]:
val_val = {
    "/users/project1/pt01299/synt/gda70/train/index_val.csv": "gda",
    "/users/project1/pt01299/synt/gda70/train/index_val_val.csv": "gda",
    "/users/project1/pt01299/synt/gda70/train/index_val_tr.csv": "gda",
    "/users/project1/pt01299/synt/segformer_dataset255_all/val/index.csv": "s",
    "/users/project1/pt01299/synt/mix_val.csv": "m",
    "/users/project1/pt01299/synt/DK/osfstorage/dataset_v2/solardk_dataset_neurips_v2/gentofte_trainval/val/index.csv": "dk"
}

def map_val(df: pd.DataFrame):
    if 'val' not in df.columns:
        df['val'] = list(val_val.keys())[0]
    v = df['val'].astype(str)
    conds = [v.str.contains(k, regex=False, na=False) for k in val_val.keys()]
    choices = list(val_val.values())
    df['val'] = select(conds, choices, default='s')
    return df

In [30]:
def add_tr_val(df: pd.DataFrame):
    df['tr_val'] = df['train'] + '_' + df['val']
    return df

## src

In [31]:
def add_src(df: pd.DataFrame, fn):
    sp = fn.split('_')
    if len(sp) > 2:
        df['src'] = sp[1] + '_' + sp[2].replace('.csv', '')
    elif 'm' in fn:
        df['src'] = 'subm'
    else:
        df['src'] = 'x'
    df['fn'] = fn
    return df

## epochs, gpus

In [32]:
with open('comb_stats_g.json', 'r') as f:
    eps1 = pd.Series(json.load(f), name='epochs_done')
with open('comb_stats.json', 'r') as f:
    eps2 = pd.Series(json.load(f), name='epochs_done')
eps = pd.concat([eps1, eps2])

In [33]:
with open('extr_gpus.json', 'r') as f:
    gpus1 = pd.Series(json.load(f), name='GPU')
with open('extr_gpus_g.json', 'r') as f:
    gpus2 = pd.Series(json.load(f), name='GPU')
gpus = pd.concat([gpus1, gpus2])
gpus = gpus.replace("NVIDIA A100 80GB PCIe", "A100")
gpus = gpus.replace("NVIDIA H100", "H100")

In [34]:
def add_eps_done(df: pd.DataFrame, eps: pd.Series):
    return pd.merge(df, eps, left_on='ID', right_index=True, how='left')

In [35]:
def add_gpus_done(df: pd.DataFrame, gpus: pd.Series):
    return pd.merge(df, gpus, left_on='ID', right_index=True, how='left')

## sub

new values

calcs valid for ph1 only

In [36]:
DS_SIZES_TR = {
    's': 374,
    'sub': 374, # s, later mult
    'dk': 324,
    'gda': 18,
}

DS_SIZES_VAL = {
    's': 80,
    'sub': 80, # s, later mult
    'dk': 160,
    'gda': 20,
    'gda_v': 2,
}

S_SUB_S_FCT = {
    'mix': 100,
    'composite': 100
}

DS_W_MULT = {
    's': 1/20,
    'sub': 1/20,
    'gda': 1,
    'dk': 1,
}

DS_DOM = {
    's': 0.75,
    'sub': 0.75,
    'gda': 1,
    'dk': 0,
}
DS_DOM = {
    's': 0.25,
    'sub': 0.25,
    'gda': 1,
    'dk': -1,
}

DS_REAL = {
    's': 0,
    'sub': 0,
    'gda': 1,
    'dk': 1,
}
DS_REAL = {
    's': -0.5, # -1,
    'sub': -0.5, # -1,
    'gda': 1,
    'dk': 1,
}

In [37]:
def add_sub_sumlt(df: pd.DataFrame) -> pd.DataFrame:
    sub_map = df['sub'].map(S_SUB_S_FCT) # mix
    df['sub_mult'] = sub_map.fillna(df['sub']) # numb
    df['sub_mult'] = df['sub_mult'].fillna(100) # NaN - no sub
    df['sub_mult'] = df['sub_mult'].astype(float) / 100

    return df

## workload

In [38]:
def add_workload(df: pd.DataFrame, fn='') -> pd.DataFrame:
    # train
    msk = df['train'].isin(['m', 'subm'])
    w_t_base = df['train'].map(DS_W_MULT) * df['train'].map(DS_SIZES_TR) * df['sub_mult'] # sub_mult == 1 for non SYNT
    w_t_m = (
        DS_W_MULT['s'] * DS_SIZES_TR['s'] * df['sub_mult'] +
        DS_W_MULT['dk'] * DS_SIZES_TR['dk']    
    )
    df['w_t'] = w_t_base.where(~msk, w_t_m)

    # val
    if (len(fn.split('_')) > 3): # ph3
        msk = df['val'] == 'gda'
        w_v_base = df['val'].map(DS_W_MULT) * df['val'].map(DS_SIZES_VAL)
        w_v_m = (
            DS_W_MULT['gda'] * DS_SIZES_VAL['gda_v']
        )
        df['w_v'] = w_v_base.where(~msk, w_v_m)
    else:
        msk = df['val'].isin(['m', 'subm'])
        w_v_base = df['val'].map(DS_W_MULT) * df['val'].map(DS_SIZES_VAL)
        w_v_m = (
            DS_W_MULT['s'] * DS_SIZES_VAL['s'] +
            DS_W_MULT['dk'] * DS_SIZES_VAL['dk']
        )
        df['w_v'] = w_v_base.where(~msk, w_v_m)

    # final
    df['workload'] = df['w_t'] + df['w_v']
    return df

## diversity

In [39]:
def add_diversity(df: pd.DataFrame) -> pd.DataFrame:
    x0, y0 = 0, 0 # ref pt zero
    
    # --- re train ---
    msk = df['train'].isin(['m', 'subm'])
    re_t_base = df['train'].map(DS_REAL)
    re_t_m_num = (
        DS_REAL['s'] * DS_SIZES_TR['s'] * df['sub_mult'] +
        DS_REAL['dk'] * DS_SIZES_TR['dk']
    )
    re_t_m_den = (
        DS_SIZES_TR['s'] * df['sub_mult'] +
        DS_SIZES_TR['dk']
    )
    re_t_m = re_t_m_num / re_t_m_den
    df['re_t'] = re_t_base.where(~msk, re_t_m)

    # --- do train ---
    do_t_base = df['train'].map(DS_DOM)
    do_t_m_num = (
        DS_DOM['s'] * DS_SIZES_TR['s'] * df['sub_mult'] +
        DS_DOM['dk'] * DS_SIZES_TR['dk']
    )
    do_t_m_den = (
        DS_SIZES_TR['s'] * df['sub_mult'] +
        DS_SIZES_TR['dk']
    )
    do_t_m = do_t_m_num / do_t_m_den
    df['do_t'] = do_t_base.where(~msk, do_t_m)

    # --- re val ---
    # df['re_v'] = df['val'].map(DS_REAL)
    msk = df['val'].isin(['m', 'subm'])
    re_v_base = df['val'].map(DS_REAL)
    re_v_m_num = (
        DS_REAL['s'] * DS_SIZES_VAL['s'] +
        DS_REAL['dk'] * DS_SIZES_VAL['dk']
    )
    re_v_m_den = (
        DS_SIZES_VAL['s'] +
        DS_SIZES_VAL['dk']
    )
    re_v_m = re_v_m_num / re_v_m_den
    df['re_v'] = re_v_base.where(~msk, re_v_m)

    df['real'] = df['re_t'] + df['re_v']

    # --- do val ---
    # df['do_v'] = df['val'].map(DS_DOM)
    do_v_base = df['val'].map(DS_DOM)
    do_v_m_num = (
        DS_DOM['s'] * DS_SIZES_VAL['s'] +
        DS_DOM['dk'] * DS_SIZES_VAL['dk']
    )
    do_v_m_den = (
        DS_SIZES_VAL['s'] +
        DS_SIZES_VAL['dk']
    )
    do_v_m = do_v_m_num / do_v_m_den
    df['do_v'] = do_v_base.where(~msk, do_v_m)

    df['dom'] = df['do_t'] + df['do_v']

    # --- euclidean distance ---
    df['dist'] = ((df['real'] - x0)**2 + (df['dom'] - y0)**2)**0.5

    return df

## try

In [40]:
f = 'dk.csv'
f = 'subs_16.csv'
# f = 'subs_m.csv'
# f = 'ph2_dk_dk.csv'
# f = 'ph1_gda.csv'
df = ld_scols(f)
df = add_tr(df, f)
df = map_val(df)
df = add_tr_val(df)
df = add_src(df, f)
df = add_sub_sumlt(df)
df = add_diversity(df)
if 'ID' not in df.columns:
        df['ID'] = 'keine'
df = add_eps_done(df, eps)
df = add_gpus_done(df, gpus)
df = add_workload(df)
df.head()

,1_epoch,sub,lrenc,mod_ph1,batch_size,ID,ema,wd,loss,epochs,...,re_v,real,do_v,dom,dist,epochs_done,GPU,w_t,w_v,workload
0,15,25,0.00005,NaN,16,dhwn3u4p,False,0.05,Dice+Focal,16,...,1.0,0.5,1.00,1.25,1.346291,14,H100,4.675,20.0,24.675
1,15,25,0.00005,NaN,16,mzs6i73s,False,0.05,Dice+Focal,16,...,-0.5,-1.0,0.25,0.50,1.118034,15,H100,4.675,4.0,8.675
2,15,15,0.00005,NaN,16,88gwxxx7,False,0.05,Dice+Focal,16,...,1.0,0.5,1.00,1.25,1.346291,15,H100,2.805,20.0,22.805
3,15,35,0.00005,NaN,16,hlikqun8,False,0.05,Dice+Focal,16,...,-0.5,-1.0,0.25,0.50,1.118034,15,H100,6.545,4.0,10.545
4,15,mix,0.00005,NaN,16,hsb5w0ul,False,0.05,Dice+Focal,16,...,-0.5,-1.0,0.25,0.50,1.118034,15,H100,18.700,4.0,22.700


## proc

In [41]:
def proc_f(f):
    df = ld_scols(f)
    df = add_tr(df, f)
    df = map_val(df)
    df = add_tr_val(df)
    df = add_src(df, f)
    df = add_sub_sumlt(df)
    df = drop_empty_cols(df)
    df = add_diversity(df)
    if 'ID' not in df.columns:
        df['ID'] = 'keine'
        print('no id in', f)
    df = add_eps_done(df, eps)
    df = add_gpus_done(df, gpus)
    df = add_workload(df)
    df.to_csv(f'{OUT_DIR}/{f}')
    
    return df

In [42]:
files = [f for f in os.listdir() if f.lower().endswith(".csv")]
files.remove('da.csv')
files

['30.csv',
 'dk.csv',
 'full.csv',
 'ph1_gda.csv',
 'ph2_dk_dk.csv',
 'ph2_dk_gda.csv',
 'ph2_sub_gda.csv',
 'ph2_sub_s.csv',
 'ph2_s_gda.csv',
 'ph2_s_s.csv',
 'ph3_subm_gda.csv',
 'ph3_subm_gda_gda.csv',
 'ph3_subm_m.csv',
 'ph3_subm_m_gda.csv',
 'ph3_subm_s.csv',
 'ph3_subm_s_gda.csv',
 'S3.csv',
 'S4.csv',
 'short.csv',
 'subs_16.csv',
 'subs_16A.csv',
 'subs_m.csv']

In [43]:
ph1 = pd.DataFrame()
ph2 = pd.DataFrame()
ph3 = pd.DataFrame()

for ff in files:
    print(ff)
    df = proc_f(ff)
    if 'ph2' in ff:
        ph2 = pd.concat([df, ph2], ignore_index=True)
    elif 'ph3' in ff:
        ph3 = pd.concat([df, ph3], ignore_index=True)
    else:
        try:
            if 'S' in ff:
                print(len(df.columns))
                print(len(pd.unique(df.columns)))
                print(df.columns)
            ph1 = pd.concat([df, ph1], ignore_index=True)
        except Exception as e:
            print('ph1', ff, e)

# ph1 = ph1.dropna(axis='columns')
ph2 = ph2.dropna(axis='columns')
ph3 = ph3.dropna(axis='columns')

ph1.to_csv(f'{OUT_DIR}/proc_ph1.csv')
ph2.to_csv(f'{OUT_DIR}/proc_ph2.csv')
ph3.to_csv(f'{OUT_DIR}/proc_ph3.csv')

30.csv
dk.csv
full.csv
ph1_gda.csv
ph2_dk_dk.csv


C:\Users\admin\AppData\Local\Temp\ipykernel_16020\2474272437.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col.replace('/', '_')] = df[col]
C:\Users\admin\AppData\Local\Temp\ipykernel_16020\2474272437.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col.replace('/', '_')] = df[col]
C:\Users\admin\AppData\Local\Temp\ipykernel_16020\2474272437.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining al

ph2_dk_gda.csv
ph2_sub_gda.csv
ph2_sub_s.csv
ph2_s_gda.csv
ph2_s_s.csv
ph3_subm_gda.csv


C:\Users\admin\AppData\Local\Temp\ipykernel_16020\2474272437.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col.replace('/', '_')] = df[col]
C:\Users\admin\AppData\Local\Temp\ipykernel_16020\2474272437.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col.replace('/', '_')] = df[col]
C:\Users\admin\AppData\Local\Temp\ipykernel_16020\2474272437.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining al

ph3_subm_gda_gda.csv
ph3_subm_m.csv


C:\Users\admin\AppData\Local\Temp\ipykernel_16020\2474272437.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col.replace('/', '_')] = df[col]
C:\Users\admin\AppData\Local\Temp\ipykernel_16020\2474272437.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col.replace('/', '_')] = df[col]
C:\Users\admin\AppData\Local\Temp\ipykernel_16020\2474272437.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining al

ph3_subm_m_gda.csv
ph3_subm_s.csv


C:\Users\admin\AppData\Local\Temp\ipykernel_16020\2474272437.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col.replace('/', '_')] = df[col]
C:\Users\admin\AppData\Local\Temp\ipykernel_16020\2474272437.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col.replace('/', '_')] = df[col]
C:\Users\admin\AppData\Local\Temp\ipykernel_16020\2474272437.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining al

ph3_subm_s_gda.csv
S3.csv
33
33
Index(['sub', 'batch_size', 'ID', 'wd', '1_epoch', 'lrdec', 'Runtime', 'Sweep',
       'loss', 'lrenc', 'epochs', 'warmup_epochs', 'val', '1_test_SYNT_iou',
       '1_test_DK_iou', '1_test_GDA_iou', 'train', 'tr_val', 'src', 'fn',
       'sub_mult', 're_t', 'do_t', 're_v', 'real', 'do_v', 'dom', 'dist',
       'epochs_done', 'GPU', 'w_t', 'w_v', 'workload'],
      dtype='str')
S4.csv
31
31
Index(['1_epoch', 'sub', 'lrenc', 'batch_size', 'ID', 'wd', 'loss',
       'warmup_epochs', 'lrdec', 'Runtime', '1_test_SYNT_iou', '1_test_DK_iou',
       '1_test_GDA_iou', 'train', 'val', 'tr_val', 'src', 'fn', 'sub_mult',
       're_t', 'do_t', 're_v', 'real', 'do_v', 'dom', 'dist', 'epochs_done',
       'GPU', 'w_t', 'w_v', 'workload'],
      dtype='str')
short.csv
subs_16.csv
subs_16A.csv
subs_m.csv
no id in subs_m.csv


drop duplicate iou cols
drop empty ious
====stack all
====join 1-2 by mod1
====join 2-3 by mod3
====generate comb_key 1-2-3
====grouped agg stats by comb_key
stack disjoint to joint - to załatwia skrypt etl, tutaj tylko prawdziwe i pełne fazy

# 2 joins + save wide

## concat 123

In [44]:
cnc = pd.concat([ph1, ph2, ph3])
cnc.head()

,1_epoch,sub,lrenc,batch_size,ema,wd,loss,epochs,warmup_epochs,lrdec,...,2_epoch,mod_ph1,2_test_GDA_iou,2_test_SYNT_iou,2_test_DK_iou,3_epoch,mod_ph2,3_test_GDA_iou,3_test_SYNT_iou,3_test_DK_iou
0,19.0,composite,0.00001,8,True,0.05,Dice+Focal,20.0,2,0.000008,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,19.0,15,0.00001,8,False,0.01,Dice+Focal,20.0,2,0.000008,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,19.0,15,0.00001,16,True,0.03,Dice+Focal,20.0,2,0.000008,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,19.0,25,0.00001,16,False,0.03,Dice+Focal,20.0,4,0.000008,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,19.0,15,0.00001,16,False,0.05,Dice+Focal,20.0,2,0.000008,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [45]:
cnc.columns

Index(['1_epoch', 'sub', 'lrenc', 'batch_size', 'ema', 'wd', 'loss', 'epochs',
       'warmup_epochs', 'lrdec', 'Runtime', 'val', 'Sweep', '1_test_SYNT_iou',
       '1_test_DK_iou', '1_test_GDA_iou', 'train', 'tr_val', 'src', 'fn',
       'sub_mult', 're_t', 'do_t', 're_v', 'real', 'do_v', 'dom', 'dist', 'ID',
       'epochs_done', 'GPU', 'w_t', 'w_v', 'workload', '2_epoch', 'mod_ph1',
       '2_test_GDA_iou', '2_test_SYNT_iou', '2_test_DK_iou', '3_epoch',
       'mod_ph2', '3_test_GDA_iou', '3_test_SYNT_iou', '3_test_DK_iou'],
      dtype='str')

In [46]:
cnc.to_csv(f'{OUT_DIR}/cnc123b.csv')

## join 12

In [47]:
df12 = pd.merge(ph1, ph2, how='inner', left_on=['ID'], right_on=['mod_ph1'])
df12.head()

,1_epoch,sub_x,lrenc_x,batch_size_x,ema_x,wd_x,loss_x,epochs_x,warmup_epochs_x,lrdec_x,...,re_v_y,real_y,do_v_y,dom_y,dist_y,epochs_done_y,GPU_y,w_t_y,w_v_y,workload_y
0,15.0,15,0.00005,16,False,0.05,Dice+Focal,16.0,6,0.000008,...,1.0,1.778611,1.000000,0.184491,1.788154,0,H100,326.805,20.0,346.805
1,15.0,15,0.00005,16,False,0.05,Dice+Focal,16.0,6,0.000008,...,-0.5,-0.303725,0.250000,-0.080229,0.314143,5,H100,342.700,4.0,346.700
2,15.0,15,0.00005,16,False,0.05,Dice+Focal,16.0,6,0.000008,...,1.0,1.778611,1.000000,0.184491,1.788154,1,H100,326.805,20.0,346.805
3,15.0,15,0.00005,16,False,0.05,Dice+Focal,16.0,6,0.000008,...,0.5,0.696275,-0.583333,-0.913563,1.148649,5,H100,342.700,164.0,506.700
4,15.0,15,0.00005,16,False,0.05,Dice+Focal,16.0,6,0.000008,...,1.0,1.778611,1.000000,0.184491,1.788154,1,H100,326.805,20.0,346.805


In [48]:
df12.columns

Index(['1_epoch', 'sub_x', 'lrenc_x', 'batch_size_x', 'ema_x', 'wd_x',
       'loss_x', 'epochs_x', 'warmup_epochs_x', 'lrdec_x', 'Runtime_x',
       'val_x', 'Sweep_x', '1_test_SYNT_iou', '1_test_DK_iou',
       '1_test_GDA_iou', 'train_x', 'tr_val_x', 'src_x', 'fn_x', 'sub_mult_x',
       're_t_x', 'do_t_x', 're_v_x', 'real_x', 'do_v_x', 'dom_x', 'dist_x',
       'ID_x', 'epochs_done_x', 'GPU_x', 'w_t_x', 'w_v_x', 'workload_x',
       '2_epoch', 'lrenc_y', 'sub_y', 'mod_ph1', 'batch_size_y', 'ID_y',
       'ema_y', 'wd_y', 'loss_y', 'epochs_y', 'warmup_epochs_y', 'lrdec_y',
       'Runtime_y', 'val_y', 'Sweep_y', '2_test_GDA_iou', '2_test_SYNT_iou',
       '2_test_DK_iou', 'train_y', 'tr_val_y', 'src_y', 'fn_y', 'sub_mult_y',
       're_t_y', 'do_t_y', 're_v_y', 'real_y', 'do_v_y', 'dom_y', 'dist_y',
       'epochs_done_y', 'GPU_y', 'w_t_y', 'w_v_y', 'workload_y'],
      dtype='str')

## join 123

In [49]:
df123 = pd.merge(df12, ph3, how='inner', left_on=['ID_y'], right_on=['mod_ph2'])
df123.head()

,1_epoch,sub_x,lrenc_x,batch_size_x,ema_x,wd_x,loss_x,epochs_x,warmup_epochs_x,lrdec_x,...,re_v,real,do_v,dom,dist,epochs_done,GPU,w_t,w_v,workload
0,15.0,mix,0.00005,8,False,0.05,Dice+Focal,16.0,6,0.000008,...,1.0,2.0,-1.0,0.0,2.0,4,A100,18.0,160.0,178.0
1,15.0,mix,0.00005,8,False,0.05,Dice+Focal,16.0,6,0.000008,...,1.0,2.0,-1.0,0.0,2.0,0,A100,18.0,160.0,178.0
2,15.0,mix,0.00005,8,False,0.05,Dice+Focal,16.0,6,0.000008,...,1.0,2.0,-1.0,0.0,2.0,1,A100,18.0,160.0,178.0
3,15.0,mix,0.00005,8,False,0.05,Dice+Focal,16.0,6,0.000008,...,1.0,2.0,1.0,0.0,2.0,2,H100,324.0,20.0,344.0
4,15.0,mix,0.00005,8,False,0.05,Dice+Focal,16.0,6,0.000008,...,1.0,2.0,1.0,0.0,2.0,2,H100,324.0,20.0,344.0


## comb, trains

In [50]:
df = df123
df['comb_key'] = df['tr_val_x'] + '|' + \
                df['tr_val_y'] + '|' + \
                df['tr_val']
df['trains'] = df['train_x'] + '_' + df['train_y'] + '_' + df['train']
df.head()

,1_epoch,sub_x,lrenc_x,batch_size_x,ema_x,wd_x,loss_x,epochs_x,warmup_epochs_x,lrdec_x,...,do_v,dom,dist,epochs_done,GPU,w_t,w_v,workload,comb_key,trains
0,15.0,mix,0.00005,8,False,0.05,Dice+Focal,16.0,6,0.000008,...,-1.0,0.0,2.0,4,A100,18.0,160.0,178.0,sub_gda|subm_m|gda_dk,sub_subm_gda
1,15.0,mix,0.00005,8,False,0.05,Dice+Focal,16.0,6,0.000008,...,-1.0,0.0,2.0,0,A100,18.0,160.0,178.0,sub_gda|subm_m|gda_dk,sub_subm_gda
2,15.0,mix,0.00005,8,False,0.05,Dice+Focal,16.0,6,0.000008,...,-1.0,0.0,2.0,1,A100,18.0,160.0,178.0,sub_gda|subm_m|gda_dk,sub_subm_gda
3,15.0,mix,0.00005,8,False,0.05,Dice+Focal,16.0,6,0.000008,...,1.0,0.0,2.0,2,H100,324.0,20.0,344.0,sub_gda|subm_m|dk_gda,sub_subm_dk
4,15.0,mix,0.00005,8,False,0.05,Dice+Focal,16.0,6,0.000008,...,1.0,0.0,2.0,2,H100,324.0,20.0,344.0,sub_gda|subm_m|dk_gda,sub_subm_dk


In [51]:
df123.columns

Index(['1_epoch', 'sub_x', 'lrenc_x', 'batch_size_x', 'ema_x', 'wd_x',
       'loss_x', 'epochs_x', 'warmup_epochs_x', 'lrdec_x',
       ...
       'do_v', 'dom', 'dist', 'epochs_done', 'GPU', 'w_t', 'w_v', 'workload',
       'comb_key', 'trains'],
      dtype='str', length=105)

## more

In [52]:
df.to_csv(f'{OUT_DIR}/ph123b.csv') # OK

# stats

In [53]:
print(ph1.size)
print(df12.size)
print(df123.size)

31858
71001
101220


In [54]:
print(cnc.size)

128920


In [55]:
df123['ID_x'].nunique()

22

In [56]:
df123['fn_x'].unique()

<ArrowStringArray>
['subs_16A.csv', 'subs_16.csv', 'short.csv', 'full.csv', 'dk.csv']
Length: 5, dtype: str

In [57]:
df123['ID_y'].nunique()

52

In [58]:
df123['ID'].nunique()

964

In [59]:
cnc['ID'].nunique()

2848

In [60]:
ph1['ID'].nunique()

855

In [61]:
ph2['ID'].nunique()

1029

In [62]:
ph3['ID'].nunique()

964